In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import numpy as np
import pandas as pd

In [3]:
data_dir = str(PROJECT_ROOT / "data" / "processed" / "TEP") + "/"
references_dir = str(PROJECT_ROOT / "references" / "TEP") + "/"
models_dir = str(PROJECT_ROOT / "models" / "TEP") + "/"
max_lag_selection_dir = models_dir + "max_lag_selection/"
weights_dir = models_dir + "max_lag_selection/weights/"
y_faultfree_dir = models_dir + "max_lag_selection/y/"
y_pred_faultfree_dir = models_dir + "max_lag_selection/y_pred/"
x_faulty_dir = models_dir + "moving_average_window_selection/X/"
y_faulty_dir = models_dir + "moving_average_window_selection/y/"
y_pred_faulty_dir = models_dir + "moving_average_window_selection/y_pred/"

In [4]:
import os

for directory in [models_dir, max_lag_selection_dir, weights_dir, y_faultfree_dir, y_pred_faultfree_dir, x_faulty_dir, y_faulty_dir, y_pred_faulty_dir]:
    os.makedirs(directory, exist_ok=True)

In [5]:
test_file_name = "TEP_Testing.parquet"
adjacency_matrix_file_name = "TEGroundTruth.txt"

In [6]:
df = pd.read_parquet(data_dir + test_file_name)

In [7]:
n_faultfree_samples = df[df["faultNumber"] == 0].shape[0]
df_faultfree = df.iloc[:n_faultfree_samples]

n_samples_per_faultfree_ts = df[df["faultNumber"] == 0].groupby(["faultNumber", "simulationRun"]).size().max()
n_faultfree_ts = df[df["faultNumber"] == 0].groupby(["sample"]).size().max()

df_faultfree = df_faultfree.drop(["faultNumber", "simulationRun", "sample"], axis=1)

In [8]:
adjacency_matrix = np.loadtxt(references_dir + adjacency_matrix_file_name)

In [9]:
feature_names = df_faultfree.columns
n_features = len(feature_names)
child_parents_pairs = {}

for child_idx in range(n_features):
    parents = []
    for feature_idx in range(n_features):
        if adjacency_matrix[feature_idx, child_idx] == 1:
            parents.append(feature_names[feature_idx])
    child_parents_pairs[feature_names[child_idx]] = parents

In [10]:
process_children = [child for child in child_parents_pairs.keys() if len(child_parents_pairs[child]) != 0 and int(child[1:]) < 23]
n_process_children = len(process_children)
process_child_parents_pairs = {child: child_parents_pairs[child] for child in process_children}

In [11]:
max_lag = 1
sample_precision = np.load(max_lag_selection_dir + "sample_precision_train.npy")

In [12]:
n_samples_test = n_faultfree_ts * (n_samples_per_faultfree_ts - max_lag)
res_test = np.empty((n_samples_test, n_process_children))

for child, parents in process_child_parents_pairs.items():

    df_X_test = df_faultfree[parents + [child]]
    y_test = df_faultfree[child]

    df_X_test_lagged = pd.concat([df_X_test.shift(i).add_suffix(f"_lag{i}") for i in range(1, max_lag + 1)], axis=1)

    mask = []
    for ts_start_idx in range(0, n_faultfree_samples, n_samples_per_faultfree_ts):
        mask.extend(range(ts_start_idx + max_lag, ts_start_idx + n_samples_per_faultfree_ts))

    df_X_test_lagged = df_X_test_lagged.iloc[mask]
    y_test = y_test.iloc[mask]

    X_test = np.ones((df_X_test_lagged.shape[0], df_X_test_lagged.shape[1] + 1))
    X_test[:, 1:] = df_X_test_lagged.values
    y_test = y_test.to_numpy()

    beta = np.load(weights_dir + f"beta_child_{child}_lag_{max_lag}.npy")

    y_test_pred = X_test @ beta

    res_test[:, process_children.index(child)] = y_test - y_test_pred   

In [13]:
scores_raw = np.mean(res_test @ sample_precision * res_test, axis=1)
scores_2d = scores_raw.reshape((n_faultfree_ts, n_samples_per_faultfree_ts - max_lag))
df_scores_2d = pd.DataFrame(scores_2d)

In [14]:
y_anomaly_2d = np.zeros_like(scores_2d, dtype=int)

In [15]:
window = 6
consecutive_alarms = 5
threshold = 1.7085420289572733

y_anomaly_2d_red = y_anomaly_2d[:, window-1:]

y_ts_anomaly = y_anomaly_2d_red.max(axis=1)
y_ts_anomaly = y_ts_anomaly.reshape(-1)

df_results = pd.DataFrame(columns=[
    "Window Size",
    "Consecutive Alarms",
    "Threshold",
    "Sample FPR",
    "Trajectory-Phase FPR",
])

scores_2d_smoothed = df_scores_2d.T.rolling(window=window).mean().values.T
scores_2d_smoothed = scores_2d_smoothed[:, window-1:]

raw_breaches = (scores_2d_smoothed > threshold).astype(int)

y_anomaly_pred_2d = (
    pd.DataFrame(raw_breaches.T).rolling(
        window=consecutive_alarms, min_periods=consecutive_alarms
    ).sum() == consecutive_alarms
).astype(int).values.T

eval_mask_2d = np.ones_like(y_anomaly_pred_2d, dtype=bool)
eval_mask_2d[:, :consecutive_alarms-1] = False

y_anomaly = y_anomaly_2d_red.reshape(-1)
y_anomaly_pred = y_anomaly_pred_2d.reshape(-1)
eval_mask = eval_mask_2d.reshape(-1)

y_anomaly_eval = y_anomaly[eval_mask]
y_anomaly_pred_eval = y_anomaly_pred[eval_mask]

fp = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 1))
tn = np.sum((y_anomaly_eval == 0) & (y_anomaly_pred_eval == 0))

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

y_ts_anomaly_pred = y_anomaly_pred_2d.max(axis=1)
y_ts_anomaly_pred = y_ts_anomaly_pred.reshape(-1)

fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))

fpr_ts = fp_ts / (fp_ts + tn_ts) if (fp_ts + tn_ts) > 0 else 0

df_result = pd.DataFrame({
    "Window Size": [window],
    "Consecutive Alarms": [consecutive_alarms],
    "Threshold": [threshold],
    "Sample FPR": [fpr],
    "Trajectory-Phase FPR": [fpr_ts],
})

df_results = pd.concat([df_results, df_result], ignore_index=True)

with pd.option_context('display.max_rows', None):
    display(df_results)

<ipykernel>\3888930378.py:60: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, df_result], ignore_index=True)


,Window Size,Consecutive Alarms,Threshold,Sample FPR,Trajectory-Phase FPR
0,6,5,1.708542,0.000225,0.09


In [16]:
from carl.tep_figure_exports import save_tep_figure_metrics
save_tep_figure_metrics(df_results, PROJECT_ROOT, 'Graph-informed', 'healthy_test', 'main_detection')


Saved TEP figure metrics to <repo>\reports\results\TEP\figure_metrics\main_detection_graph_informed_healthy_test.csv
